FEATURE EXTRACTION & CHAPTER GENERATION

Textual & Visual Feature Extraction → Lightweight LLM → JSON Chapters

CPU-Optimized for Intel i7-8550U, 16GB RAM

CONFIGURATION - Enter Filepaths

In [1]:
# ==================== CONFIGURATION ====================
VIDEO_PATH = r"C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\static\videos\Week 02 - Embedded S.mp4"
TRANSCRIPT_PATH = r"C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\Transcripts\Week 02 - Embedded S_transcript.txt"
OUTPUT_DIR = r"C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\Output"

# Processing Parameters
FRAME_SAMPLE_RATE = 60        # Extract 1 frame every N seconds (higher = faster)
WORDS_PER_CHAPTER = 200      # Words per chapter segment (affects granularity)
USE_LIGHTWEIGHT_VISUAL = True # Use ResNet18 instead of ResNet50 (faster)
# =======================================================


IMPORTS

In [2]:
# %pip install hf_transfer
import os
import re
import json
import warnings
from pathlib import Path
from typing import List, Dict, Any, Tuple
from datetime import datetime

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"
os.environ["HF_DATASETS_AUDIO_BACKEND"] = "soundfile"

# Computer Vision
import cv2
import numpy as np
from PIL import Image

# Deep Learning
import torch
import torch.nn as nn
from torchvision import transforms, models

# NLP & Embeddings  
from sentence_transformers import SentenceTransformer

# LLM
from transformers.models.auto.tokenization_auto import AutoTokenizer
from transformers.models.auto.modeling_auto import AutoModelForCausalLM
# %pip install bitsandbytes
# %pip install sentencepiece
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from transformers.utils.quantization_config import BitsAndBytesConfig


# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


✅ All imports successful
PyTorch version: 2.10.0+cpu
CUDA available: False
Device: CPU


UTILITY FUNCTIONS

In [3]:
def convert_timestamp_to_datetime(time_str: str) -> datetime:
    """Parse time string to datetime object (supports HH:MM:SS)."""
    time_obj = datetime.strptime(time_str, "%H:%M:%S.%f")
    return time_obj

def parse_time_string(time_str: str) -> float:
    """Parse time string to seconds (supports HH:MM:SS)."""
    time_obj = datetime.strptime(time_str, "%H:%M:%S.%f").time()
    parsed_time = 0.0
    try:
        parsed_time = time_obj.hour * 3600 + \
                time_obj.minute * 60 + \
                time_obj.second + \
                time_obj.microsecond / 1e6
    except:
        return 0.0
    
    return parsed_time

def seconds_to_time_str(seconds: float) -> str:
    """Convert seconds to HH:MM:SS format."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

"""
FUNCTION: To clean the filler words from the transcript
INPUT: str (i.e. text from transcript)
OUTPUT: str (cleaned text)
"""
def clean_filler_words(line: str) -> str:
    # List of filler word upto the Developer's knowledge
    filler_words = ["um", "uh...", "yeah.", "um...", "uh", "...", "like", "you know", "so", "actually", "basically", "cool", "I mean"]
    line = line.lower()
    for word in filler_words:
        line = line.replace(word+" ", "")
        if line.endswith(word):
            line = line[:-len(word)]

    return line.strip("\n")

"""
Load time-based transcript from text file.
Expected format:
    [HH:MM:SS.mmm --> HH:MM:SS.mmm]
    Text line
    
    [HH:MM:SS.mmm --> HH:MM:SS.mmm]
    Another text line

OUTPUT: List of segments with start_time, end_time, text
"""
def load_transcript(file_path: str) -> List[Dict[str, Any]]:

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    segments = []
    i = 0
    
    while i < len(lines):
        line = lines[i].strip()
        
        # Look for timestamp line with "-->"
        if "-->" in line:
            # Remove brackets if present
            line = line.strip('[]')
            
            try:
                # Split by -->
                parts = line.split('-->')
                if len(parts) == 2:
                    start_str = parts[0].strip()
                    end_str = parts[1].strip()
                    
                    # Convert to datetime objects
                    start_time_obj = convert_timestamp_to_datetime(start_str)
                    end_time_obj = convert_timestamp_to_datetime(end_str)
                    
                    # Convert to seconds
                    start_seconds = (start_time_obj.hour * 3600 + 
                                   start_time_obj.minute * 60 + 
                                   start_time_obj.second + 
                                   start_time_obj.microsecond / 1e6)
                    
                    end_seconds = (end_time_obj.hour * 3600 + 
                                 end_time_obj.minute * 60 + 
                                 end_time_obj.second + 
                                 end_time_obj.microsecond / 1e6)
                    
                    # Get the text from next line(s)
                    i += 1
                    text_lines = []
                    
                    # Collect text until we hit another timestamp or empty line
                    while i < len(lines):
                        current_line = lines[i].strip()
                        
                        # Stop if we hit another timestamp or blank line
                        if "-->" in current_line or len(current_line) == 0:
                            break
                        
                        text_lines.append(current_line)
                        i += 1
                    
                    # Join all text lines
                    text = " ".join(text_lines)
                    
                    # Clean filler words
                    cleaned_text = clean_filler_words(text)
                    
                    # Only add if there's actual content
                    if len(cleaned_text) > 1:
                        segments.append({
                            'start_time': start_seconds,
                            'end_time': end_seconds,
                            'text': cleaned_text
                        })
                    
                    # Continue from current position
                    continue
                    
            except Exception as e:
                print(f"⚠️ Error parsing line {i}: {line}")
                print(f"   Error: {e}")
        
        i += 1
    
    return segments


print("✅ Utility functions loaded")


✅ Utility functions loaded


STEP 1: Load Transcript from File

In [4]:
print("="*60)
print("LOADING TRANSCRIPT")
print("="*60)

# Verify files exist
if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")
if not os.path.exists(TRANSCRIPT_PATH):
    raise FileNotFoundError(f"Transcript not found: {TRANSCRIPT_PATH}")

print(f"Video: {VIDEO_PATH}")
print(f"Transcript: {TRANSCRIPT_PATH}")

# Load transcript
transcript_segments = load_transcript(TRANSCRIPT_PATH)

print(f"\n✅ Loaded {len(transcript_segments)} transcript segments")
if transcript_segments:
    print(f"\nFirst segment:")
    seg = transcript_segments[0]
    print(f"  Time: {seconds_to_time_str(seg['start_time'])} → {seconds_to_time_str(seg['end_time'])}")
    print(f"  Text: {seg['text'][:100]}")


LOADING TRANSCRIPT
Video: C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\static\videos\Week 02 - Embedded S.mp4
Transcript: C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\Transcripts\Week 02 - Embedded S_transcript.txt

✅ Loaded 1013 transcript segments

First segment:
  Time: 00:00:15 → 00:00:23
  Text: okay, good morning. as i said a bit earlier, i'm philip lung and you guys are stuck with


STEP 2: Create Chapter Chunks

In [5]:
def create_chapters(
    segments: List[Dict[str, Any]],
    words_per_chapter: int = 100
) -> List[Dict[str, Any]]:
    """
    Combine transcript segments into logical chapters.
    """
    chapters = []
    current_words = []
    current_start = None
    current_end = None
    
    for segment in segments:
        words = segment['text'].split()
        
        if current_start is None:
            current_start = segment['start_time']
        
        current_words.extend(words)
        current_end = segment['end_time']
        
        if len(current_words) >= words_per_chapter:
            chapters.append({
                'start_time': current_start,
                'end_time': current_end,
                'text': ' '.join(current_words),
                'word_count': len(current_words)
            })
            current_words = []
            current_start = None
    
    # Add remaining words
    if current_words:
        chapters.append({
            'start_time': current_start,
            'end_time': current_end,
            'text': ' '.join(current_words),
            'word_count': len(current_words)
        })
    
    return chapters


print("="*60)
print("CREATING CHAPTERS")
print("="*60)

chapters = create_chapters(transcript_segments, words_per_chapter=WORDS_PER_CHAPTER)

print(f"✅ Created {len(chapters)} chapters")
print(f"Average words per chapter: {np.mean([c['word_count'] for c in chapters]):.1f}")

if chapters:
    print(f"\nFirst chapter:")
    ch = chapters[0]
    print(f"  Time: {seconds_to_time_str(ch['start_time'])} → {seconds_to_time_str(ch['end_time'])}")
    print(f"  Words: {ch['word_count']}")
    print(f"  Text: {ch['text'][:700]}...")


CREATING CHAPTERS
✅ Created 57 chapters
Average words per chapter: 208.0

First chapter:
  Time: 00:00:15 → 00:01:55
  Words: 211
  Text: okay, good morning. as i said a bit earlier, i'm philip lung and you guys are stuck with me this semester for elec 3607 and just want to let if anybody has any questions about this course or any concerns, whatever, i will normally be in the first half of every lab session on mondays and tuesdays and that's a good opportunity to come and have a chat with me. alternatively, you can always email me and make an appointment and then we can have a chat as well. i was told by lewis that he didn't quite finish all the parts of this, he didn't quite answer these four questions that were at the end of the lecture last week i just want to quickly go through these and you guys let me know if you have an...


STEP 3: Extract Visual Features (CPU-Optimized)

In [6]:
class VisualFeatureExtractor:
    """Efficient visual feature extraction using ResNet."""
    
    def __init__(self, use_lightweight: bool = True):
        print(f"Loading ResNet model...")
        
        # Choose model based on speed/quality tradeoff
        if use_lightweight:
            # ResNet18: Faster, lower memory (11M params)
            self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            print("  Using ResNet18 (lightweight, faster)")
        else:
            # ResNet50: Better quality, slower (25M params)
            self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            print("  Using ResNet50 (higher quality)")
        
        # Remove classification layer to get features
        self.model = nn.Sequential(*list(self.model.children())[:-1])
        self.model.eval()
        
        # Preprocessing
        self.preprocess = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])
        
        print("✅ Model loaded")
    
    def extract_from_video(
        self,
        video_path: str,
        sample_rate: int = 30
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Extract visual features from video.
        
        Args:
            video_path: Path to video file
            sample_rate: Extract 1 frame every N seconds
        
        Returns:
            (features, timestamps) as numpy arrays
        """
        print(f"\nOpening video: {video_path}")
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {video_path}")
        
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps
        
        print(f"Video info:")
        print(f"  FPS: {fps:.2f}")
        print(f"  Total frames: {total_frames}")
        print(f"  Duration: {seconds_to_time_str(duration)}")
        print(f"  Sample rate: 1 frame every {sample_rate}s")
        
        features = []
        timestamps = []
        frame_count = 0
        frames_per_sample = int(fps * sample_rate)
        
        print("\nExtracting features...")
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % frames_per_sample == 0:
                current_time = frame_count / fps
                
                # Convert BGR to RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                # Preprocess
                preprocessed = self.preprocess(frame_rgb)
                input_tensor = preprocessed.unsqueeze(0)
                
                # Extract features
                with torch.no_grad():
                    feature = self.model(input_tensor).squeeze().numpy()
                
                features.append(feature)
                timestamps.append(current_time)
                
                # Progress
                if len(features) % 20 == 0:
                    progress = (frame_count / total_frames) * 100
                    print(f"  {progress:.1f}% - {len(features)} features", end='\r')
            
            frame_count += 1
        
        cap.release()
        
        features = np.array(features)
        timestamps = np.array(timestamps)
        
        print(f"\n✅ Extracted {len(features)} visual features")
        print(f"Feature dimension: {features.shape[1]}")
        
        return features, timestamps


print("="*60)
print("EXTRACTING VISUAL FEATURES")
print("="*60)

visual_extractor = VisualFeatureExtractor(use_lightweight=USE_LIGHTWEIGHT_VISUAL)
visual_features, visual_timestamps = visual_extractor.extract_from_video(
    VIDEO_PATH,
    # sample_rate=FRAME_SAMPLE_RATE
)


EXTRACTING VISUAL FEATURES
Loading ResNet model...
  Using ResNet18 (lightweight, faster)
✅ Model loaded

Opening video: C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\static\videos\Week 02 - Embedded S.mp4
Video info:
  FPS: 30.00
  Total frames: 207378
  Duration: 01:55:12
  Sample rate: 1 frame every 30s

Extracting features...
  95.0% - 220 features
✅ Extracted 231 visual features
Feature dimension: 512


STEP 4: Generate Text Embeddings

In [7]:
print("="*60)
print("GENERATING TEXT EMBEDDINGS")
print("="*60)

print("Loading SentenceTransformer model...")
# Use lightweight model optimized for CPU
text_model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim, 80MB, fast
print("✅ Model loaded")

print("\nGenerating embeddings for chapters...")
chapter_texts = [ch['text'] for ch in chapters]
text_embeddings = text_model.encode(
    chapter_texts,
    show_progress_bar=True,
    batch_size=8  # Small batch for CPU
)

print(f"\n✅ Generated {len(text_embeddings)} text embeddings")
print(f"Embedding dimension: {text_embeddings.shape[1]}")


GENERATING TEXT EMBEDDINGS
Loading SentenceTransformer model...
✅ Model loaded

Generating embeddings for chapters...


Batches: 100%|██████████| 8/8 [00:21<00:00,  2.72s/it]


✅ Generated 57 text embeddings
Embedding dimension: 384


STEP 5: Align Visual Features with Chapters

In [9]:
def align_visual_features(
    chapters: List[Dict[str, Any]],
    visual_features: np.ndarray,
    visual_timestamps: np.ndarray
) -> List[Dict[str, Any]]:
    """
    Align visual features with chapters based on timestamps.
    For each chapter, find matching visual frames and average them.
    """
    aligned = []
    
    for chapter in chapters:
        start = chapter['start_time']
        end = chapter['end_time']
        
        # Find frames within time window
        mask = (visual_timestamps >= start) & (visual_timestamps <= end)
        matching_features = visual_features[mask]
        
        if len(matching_features) > 0:
            # Average all matching frames
            avg_feature = np.mean(matching_features, axis=0)
        else:
            # No frames in window - use nearest
            nearest_idx = np.argmin(np.abs(visual_timestamps - start))
            avg_feature = visual_features[nearest_idx]
        
        aligned.append({
            **chapter,
            'visual_feature': avg_feature
        })
    
    return aligned


print("="*60)
print("ALIGNING VISUAL FEATURES")
print("="*60)

chapters_with_visual = align_visual_features(
    chapters,
    visual_features,
    visual_timestamps
)

print(f"✅ Aligned visual features with {len(chapters_with_visual)} chapters")


ALIGNING VISUAL FEATURES
✅ Aligned visual features with 57 chapters


STEP 6: Combine All Features into chapter_features

In [11]:
print("="*60)
print("CREATING chapter_features")
print("="*60)

chapter_features = []

for idx, chapter in enumerate(chapters_with_visual):
    feature_dict = {
        'chapter_id': idx,
        'start_time': seconds_to_time_str(chapter['start_time']),
        'end_time': seconds_to_time_str(chapter['end_time']),
        'start_seconds': float(chapter['start_time']),
        'end_seconds': float(chapter['end_time']),
        'duration_seconds': float(chapter['end_time'] - chapter['start_time']),
        'text': chapter['text'],
        'word_count': chapter['word_count'],
        
        # Features for LLM input
        'text_embedding': text_embeddings[idx].tolist(),
        'visual_feature': chapter['visual_feature'].tolist(),
        
        # Metadata
        'text_embedding_dim': len(text_embeddings[idx]),
        'visual_feature_dim': len(chapter['visual_feature'])
    }
    
    chapter_features.append(feature_dict)

print(f"✅ Created {len(chapter_features)} chapter_features")

if chapter_features:
    sample = chapter_features[0]
    print(f"\nSample chapter_features structure:")
    print(f"  Keys: {list(sample.keys())}")
    print(f"  Chapter ID: {sample['chapter_id']}")
    print(f"  Time: {sample['start_time']} → {sample['end_time']}")
    print(f"  Text embedding dim: {sample['text_embedding_dim']}")
    print(f"  Visual feature dim: {sample['visual_feature_dim']}")


CREATING chapter_features
✅ Created 57 chapter_features

Sample chapter_features structure:
  Keys: ['chapter_id', 'start_time', 'end_time', 'start_seconds', 'end_seconds', 'duration_seconds', 'text', 'word_count', 'text_embedding', 'visual_feature', 'text_embedding_dim', 'visual_feature_dim']
  Chapter ID: 0
  Time: 00:00:15 → 00:01:55
  Text embedding dim: 384
  Visual feature dim: 512


STEP 7: Load Lightweight LLM (Qwe2-1.5B)

In [12]:
from huggingface_hub import hf_hub_download
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"


print("="*60)
print("LOADING LIGHTWEIGHT LLM")
print("="*60)

# Using Phi-3-mini: 3.8B params, optimized for CPU, high quality
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# model_name = "Phi-3-mini"
# model_name = "NousResearch/Phi-3-mini-1T-1.5B-Chat-v0.1"
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# model_name = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading: {model_name}")
print("This may take a few minutes on first run...\n")

# Load tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
print("✅ Tokenizer loaded")

# Load model with CPU optimizations
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # CPU requires float32
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
llm_model.eval()

print("✅ LLM loaded successfully")
print(f"Model: {model_name}")
print(f"Parameters: ~3.8B")
print(f"Device: CPU")


LOADING LIGHTWEIGHT LLM
Loading: Qwen/Qwen2.5-1.5B-Instruct
This may take a few minutes on first run...

✅ Tokenizer loaded


KeyboardInterrupt: 

STEP 8: Generate JSON Chapters with LLM

Output Format: `{"chapter": str, "start_time": str, "end_time": str, "description": str}`

Model Name                                              Time taken to generate output
Qwen2-1.5B-Instruct                                             86min
Qwen2.5-3B-Instruct                                             300min (79 chapters - avg. 201 words/chapter)

In [ ]:
import re

def generate_chapter_json(
    chapter_feature: Dict[str, Any],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    max_retries: int = 3  # bumped to 3
) -> Dict[str, str]:

    prompt = f"""Generate a chapter summary JSON object for this video segment.

    Segment Information:
    - Time Range: {chapter_feature['start_time']} to {chapter_feature['end_time']}
    - Duration: {chapter_feature['duration_seconds']:.1f} seconds
    - Transcript: {chapter_feature['text'][:500]}

    Return ONLY a single valid JSON object with exactly these fields:
    {{
    "ID": "{chapter_feature['chapter_id'] + 1}",
    "chapter": "<descriptive title, 5-8 words>",
    "start_time": "{chapter_feature['start_time']}",
    "end_time": "{chapter_feature['end_time']}",
    "description": "<1-2 sentence summary>",
    "key": "True or False"
    }}

    Rules:
    - Output ONLY the JSON object, nothing else
    - No markdown, no code fences, no explanation
    - Keep description under 50 words
    - key should be "True" if this JSON object is a key moment (important for understanding the video), otherwise "False"
    - ID must be the string "{chapter_feature['chapter_id'] + 1}"
    """

    for attempt in range(max_retries):
        generated = ""
        try:
            messages = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            inputs = tokenizer(formatted, return_tensors="pt")

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=300,       # reduced — chapters don't need 600 tokens
                    temperature=0.1,          # lower = more deterministic JSON
                    do_sample=True,
                    top_p=0.9,
                    repetition_penalty=1.1,   # discourages rambling after closing }
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            generated = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True
            ).strip()

            # --- Robust cleaning ---

            # 1. Strip markdown fences
            generated = re.sub(r"```(?:json)?", "", generated).strip()

            # 2. Extract the FIRST complete JSON object only
            #    This handles trailing text / second objects / cut-off content
            brace_count = 0
            start_idx = None
            end_idx = None
            for i, ch in enumerate(generated):
                if ch == "{":
                    if start_idx is None:
                        start_idx = i
                    brace_count += 1
                elif ch == "}":
                    brace_count -= 1
                    if brace_count == 0 and start_idx is not None:
                        end_idx = i + 1
                        break  # stop at first complete object

            if start_idx is None or end_idx is None:
                raise ValueError(f"No complete JSON object found in output")

            json_str = generated[start_idx:end_idx]

            # 3. Fix common model quirks before parsing
            #    - Trailing commas before closing brace: {"a": 1,}
            json_str = re.sub(r",\s*}", "}", json_str)
            #    - Unquoted True/False/None → JSON booleans-as-strings
            json_str = re.sub(r':\s*True\b', ': "True"', json_str)
            json_str = re.sub(r':\s*False\b', ': "False"', json_str)

            result = json.loads(json_str)

            # 4. Normalise types — model sometimes outputs ID as int
            result["ID"] = str(result.get("ID", chapter_feature['chapter_id'] + 1))
            result["key"] = str(result.get("key", "True"))

            # 5. Enforce correct timestamps (model sometimes hallucinates these)
            result["start_time"] = chapter_feature['start_time']
            result["end_time"] = chapter_feature['end_time']

            required = ['ID', 'chapter', 'start_time', 'end_time', 'description', 'key']
            if all(field in result for field in required):
                return result
            else:
                missing = [f for f in required if f not in result]
                print(f"  ⚠️ Missing fields {missing}, retry {attempt + 1}/{max_retries}")

        except json.JSONDecodeError as e:
            print(f"  ⚠️ JSON parse error (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt == max_retries - 1:
                print(f"  Raw output: {generated[:300]}")

        except Exception as e:
            print(f"  ⚠️ Error (attempt {attempt + 1}/{max_retries}): {e}")

    # Fallback
    print("  ❌ Using fallback")
    return {
        "ID": str(chapter_feature['chapter_id'] + 1),
        "chapter": f"Chapter {chapter_feature['chapter_id'] + 1}",
        "start_time": chapter_feature['start_time'],
        "end_time": chapter_feature['end_time'],
        "description": chapter_feature['text'][:200].strip(),
        "key": "True"
    }



print("="*60)
print("GENERATING CHAPTER JSON WITH LLM")
print("="*60)
print(f"Processing {len(chapter_features)} chapters...\n")

# Generate chapters
generated_chapters = []

for idx, feature in enumerate(chapter_features):
    print(f"[{idx + 1}/{len(chapter_features)}] Processing chapter {feature['chapter_id']+1}...")
    print(f"  Time: {feature['start_time']} → {feature['end_time']}")
    
    chapter_json = generate_chapter_json(feature, llm_model, llm_tokenizer)
    generated_chapters.append(chapter_json)
    
    print(f"  ✅ {chapter_json['chapter']}")
    print()

print(f"✅ Generated {len(generated_chapters)} chapter JSONs")


STEP 9: Save Results

In [ ]:
print("="*60)
print("SAVING RESULTS")
print("="*60)

# Save chapter JSONs (main output)
chapters_output = os.path.join(OUTPUT_DIR, "shortened-chapters.json")
with open(chapters_output, 'w', encoding='utf-8') as f:
    json.dump(generated_chapters, f, indent=2, ensure_ascii=False)
print(f"✅ Chapters: {chapters_output}")

# Save full chapter_features (with embeddings)
features_output = os.path.join(OUTPUT_DIR, "shortened-chapter_features_full.json")
with open(features_output, 'w', encoding='utf-8') as f:
    json.dump(chapter_features, f, indent=2, ensure_ascii=False)
print(f"✅ Full features: {features_output}")

# Create summary report
report_output = os.path.join(OUTPUT_DIR, "shortened-chapters_report.txt")
with open(report_output, 'w', encoding='utf-8') as f:
    f.write("VIDEO CHAPTER GENERATION REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(f"Video: {VIDEO_PATH}\n")
    f.write(f"Transcript: {TRANSCRIPT_PATH}\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total Chapters: {len(generated_chapters)}\n")
    f.write("\n" + "="*80 + "\n\n")
    
    for idx, chapter in enumerate(generated_chapters):
        f.write(f"CHAPTER {idx + 1}\n")
        f.write("-" * 80 + "\n")
        f.write(f"Title: {chapter['chapter']}\n")
        f.write(f"Time: {chapter['start_time']} → {chapter['end_time']}\n")
        f.write(f"\nDescription:\n{chapter['description']}\n")
        f.write("\n" + "="*80 + "\n\n")

print(f"✅ Report: {report_output}")

print("\n" + "="*60)
print("PROCESSING COMPLETE!")
print("="*60)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nGenerated files:")
print(f"  1. chapters.json - Main output (chapter JSONs)")
print(f"  2. chapter_features_full.json - Features + embeddings")
print(f"  3. chapters_report.txt - Human-readable report")


SAVING RESULTS
✅ Chapters: C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\Output\shortened-chapters.json
✅ Full features: C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\Output\shortened-chapter_features_full.json
✅ Report: C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\Output\shortened-chapters_report.txt

PROCESSING COMPLETE!

Output directory: C:\Users\amitt\VIDEO-AI-NLP-and-Deep-Learning-Applications\Output

Generated files:
  1. chapters.json - Main output (chapter JSONs)
  2. chapter_features_full.json - Features + embeddings
  3. chapters_report.txt - Human-readable report


STEP 10: Display Sample Results

In [29]:
if generated_chapters:
    print("\n" + "="*80)
    print("SAMPLE GENERATED CHAPTERS")
    print("="*80)
    
    # Show first 3 chapters
    for idx, chapter in enumerate(generated_chapters[:3]):
        print(f"\nCHAPTER {idx + 1}")
        print("-" * 80)
        print(json.dumps(chapter, indent=2))
    
    print("\n" + "="*80)
    print(f"\nTotal: {len(generated_chapters)} chapters generated")
    print(f"See {os.path.join(OUTPUT_DIR, 'shortened-chapters.json')} for complete output")



SAMPLE GENERATED CHAPTERS

CHAPTER 1
--------------------------------------------------------------------------------
{
  "ID": "1",
  "chapter": "Introduction to Embedded Systems",
  "start_time": "00:00:18",
  "end_time": "00:02:11",
  "description": "Philip introduces himself as the instructor for two courses on embedded systems.",
  "key": "True"
}

CHAPTER 2
--------------------------------------------------------------------------------
{
  "ID": "2",
  "chapter": "Lab Schedule & Assessment Structure",
  "start_time": "00:02:12",
  "end_time": "00:03:32",
  "description": "The speaker advises not emailing him directly but instead contacting Philip for coordination. Students are meeting their tutors Ellen and Patrick in lab this week, which includes weekly assessments including quizzes.",
  "key": "True"
}

CHAPTER 3
--------------------------------------------------------------------------------
{
  "ID": "3",
  "chapter": "Project Implementation & Reporting",
  "start_time": "0

STEP 11: Retrieve High-Level & Low-level Chapters

In [ ]:
#Prompt LLM for high level chapters

def generate_high_level_chapters(chapter_features: List[Dict[str, Any]], model, tokenizer) -> Dict[str, str]:
    """
    Generate high-level chapters using LLM.
    
    Returns:
        {
            "high_level": [ {chapter_json}, ... ],
            "low_level": [ {chapter_json}, ... ]
        }
    """
    high_level_chapters = []
    
    # Create prompt for high-level chapter
    high_prompt = f"""Generate a HIGH-LEVEL chapters title and description for this video segment.

        Feature: {chapter_features}
        Rules:
        - Focus on the main theme or topic of the segment
        - Output format:
        {{
        "chapter": "High-level chapter title (5-8 words)",
        "start_time": {feature['start_time']},
        "end_time": {feature['énd_time']},
        "description": "Brief 1-2 line description"
        }}
    """

    return json.loads(high_prompt)

chapters = [i for i in f.open("Output\chapters.json")]

# Correct path (use raw string or double backslashes)
with open(r"Output\chapters.json", "r", encoding="utf-8") as f:
    chapters = json.load(f)
high_level_chaps = generate_high_level_chapters(chapters, model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer)


SyntaxError: invalid syntax (365268031.py, line 32)

## 📝 Usage Notes

### Output Format
Each chapter in `chapters.json` has this structure:
```json
{
  "chapter": "Introduction to Neural Networks",
  "start_time": "00:00:12",
  "end_time": "00:02:45",
  "description": "Overview of neural network fundamentals and key concepts."
}
```

### Performance Tuning

**Speed up processing:**
- Increase `FRAME_SAMPLE_RATE` (e.g., 5 or 10 seconds)
- Set `USE_LIGHTWEIGHT_VISUAL = True` (ResNet18)
- Increase `WORDS_PER_CHAPTER` (fewer, longer chapters)

**Improve quality:**
- Decrease `FRAME_SAMPLE_RATE` (1-2 seconds)
- Set `USE_LIGHTWEIGHT_VISUAL = False` (ResNet50)
- Decrease `WORDS_PER_CHAPTER` (more, shorter chapters)

### Expected Processing Times (10-min video)
- Visual features: 3-5 minutes
- Text embeddings: 10-30 seconds
- LLM generation: 2-3 minutes per chapter

**Total:** ~70mins



In [ ]:
import flask

#Flask UI displaying the video with generated chapters
generated_chapters = 'VIDEO-AI-NLP-and-Deep-Learning-Applications\Output\chapters.json'
def UI(video_path, chapters):
    app = flask.Flask(__name__)

    @app.route('/')
    def index():
        # Generate HTML for chapters
        chapters_html = ""
        for ch in chapters:
            chapters_html += f"<li><strong>{ch['chapter']}</strong> ({ch['start_time']} - {ch['end_time']}): {ch['description']}</li>"
        
        html_content = f"""
        <html>
        <head>
            <title>Video Chapters</title>
        </head>
        <body>
            <h1>Video Chapters</h1>
            <video width="640" controls>
                <source src="{video_path}" type="video/mp4">
                Your browser does not support the video tag.
            </video>
            <h2>Chapters</h2>
            <ul>
                {chapters_html}
            </ul>
        </body>
        </html>
        """
        return html_content

    app.add_url_rule('/', 'index', index)
    app.run(host='0.0.0.0', port=5000, debug=True)

UI(VIDEO_PATH, generated_chapters)



   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------------------------------------- 0/4 [werkzeug]
   ---------- ----------------------------- 1/4 [itsdangerous]
   ---------- ----------------------------- 1/4 [itsdangerous]
   ------------------------------ --------- 3/4 [flask]
   ------------------------------ --------- 3/4 [flask]
   ------------------------------ --------- 3/4 [flask]
   ------------------------------ --------- 3/4 [flask]



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
